# 01 — Olist Data-Quality Audit

This notebook is the **presentation layer** for the source-data audit. All discovery, validation, and report generation run in reusable Python modules through:

```powershell
python scripts\run_data_quality_audit.py
```

The notebook reads the generated reports; it does not modify or re-audit the raw CSV files.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError('Could not locate the project root containing src/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import REPORTS_DIR

AUDIT_REPORT_DIR = REPORTS_DIR / 'data_quality'
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## Load generated audit reports

The validation below provides a clear instruction if the command-line pipeline has not been run yet.

In [2]:
required_reports = {
    'overview': AUDIT_REPORT_DIR / 'table_overview.csv',
    'columns': AUDIT_REPORT_DIR / 'column_quality.csv',
    'timestamps': AUDIT_REPORT_DIR / 'timestamp_issues.csv',
    'issues': AUDIT_REPORT_DIR / 'issue_register.csv',
    'schemas': AUDIT_REPORT_DIR / 'schema_checks.csv',
    'keys': AUDIT_REPORT_DIR / 'key_checks.csv',
    'relationships': AUDIT_REPORT_DIR / 'relationship_checks.csv',
    'business_rules': AUDIT_REPORT_DIR / 'business_rule_checks.csv',
    'summary': AUDIT_REPORT_DIR / 'audit_summary.json',
}
missing_reports = [path for path in required_reports.values() if not path.is_file()]
if missing_reports:
    raise FileNotFoundError(
        'Audit reports are missing. From the repository root, run: '
        'python scripts\\run_data_quality_audit.py'
    )

overview = pd.read_csv(required_reports['overview'])
column_quality = pd.read_csv(required_reports['columns'])
timestamp_issues = pd.read_csv(required_reports['timestamps'])
issue_register = pd.read_csv(required_reports['issues'])
schema_checks = pd.read_csv(required_reports['schemas'])
key_checks = pd.read_csv(required_reports['keys'])
relationship_checks = pd.read_csv(required_reports['relationships'])
business_rule_checks = pd.read_csv(required_reports['business_rules'])
with required_reports['summary'].open(encoding='utf-8') as summary_file:
    audit_summary = json.load(summary_file)

print(
    f"Audit generated: {audit_summary['generated_at_utc']} | "
    f"Files audited: {audit_summary['audited_file_count']} | "
    f"Load errors: {audit_summary['load_error_count']}"
)

Audit generated: 2026-09-01T15:46:03.293203+00:00 | Files audited: 9 | Load errors: 0


## Table overview

This is a structural comparison across source tables. Row counts are table-specific and should not be added together as a business KPI.

In [3]:
display(overview.sort_values('rows', ascending=False).reset_index(drop=True))

,file_name,rows,columns,duplicate_rows,columns_with_missing,likely_primary_keys,quality_warnings
0,olist_geolocation_dataset.csv,1000163,5,261831,0,None identified,2
1,olist_order_items_dataset.csv,112650,7,0,0,None identified,1
2,olist_order_payments_dataset.csv,103886,5,0,0,None identified,1
3,olist_customers_dataset.csv,99441,5,0,0,customer_id,0
4,olist_orders_dataset.csv,99441,8,0,3,"order_id, customer_id",1
5,olist_order_reviews_dataset.csv,99224,7,0,2,None identified,2
6,olist_products_dataset.csv,32951,9,0,8,product_id,1
7,olist_sellers_dataset.csv,3095,4,0,0,seller_id,0
8,product_category_name_translation.csv,71,2,0,0,None identified,1


## Missingness by column

Only columns with at least one missing value are shown. Missing values require business interpretation before any cleaning decision.

In [4]:
columns_with_missing = (
    column_quality.loc[column_quality['missing_values'].gt(0)]
    .sort_values(['missing_percent', 'file_name'], ascending=[False, True])
    .reset_index(drop=True)
)
if columns_with_missing.empty:
    print('No missing values were detected.')
else:
    display(columns_with_missing)

,file_name,column_name,inferred_dtype,missing_values,missing_percent,likely_primary_key
0,olist_order_reviews_dataset.csv,review_comment_title,object,87656,88.3415,False
1,olist_order_reviews_dataset.csv,review_comment_message,object,58247,58.7025,False
2,olist_orders_dataset.csv,order_delivered_customer_date,object,2965,2.9817,False
3,olist_products_dataset.csv,product_category_name,object,610,1.8512,False
4,olist_products_dataset.csv,product_name_lenght,float64,610,1.8512,False
5,olist_products_dataset.csv,product_description_lenght,float64,610,1.8512,False
6,olist_products_dataset.csv,product_photos_qty,float64,610,1.8512,False
7,olist_orders_dataset.csv,order_delivered_carrier_date,object,1783,1.7930,False
8,olist_orders_dataset.csv,order_approved_at,object,160,0.1609,False
9,olist_products_dataset.csv,product_weight_g,float64,2,0.0061,False


## Duplicate rows and candidate keys

Candidate keys are heuristic suggestions, not database constraints. The database-design phase must confirm table grain and composite keys.

In [5]:
display(
    overview[[
        'file_name', 'duplicate_rows', 'likely_primary_keys', 'quality_warnings'
    ]].sort_values(['duplicate_rows', 'file_name'], ascending=[False, True])
)

,file_name,duplicate_rows,likely_primary_keys,quality_warnings
1,olist_geolocation_dataset.csv,261831,None identified,2
0,olist_customers_dataset.csv,0,customer_id,0
2,olist_order_items_dataset.csv,0,None identified,1
3,olist_order_payments_dataset.csv,0,None identified,1
4,olist_order_reviews_dataset.csv,0,None identified,2
5,olist_orders_dataset.csv,0,"order_id, customer_id",1
6,olist_products_dataset.csv,0,product_id,1
7,olist_sellers_dataset.csv,0,seller_id,0
8,product_category_name_translation.csv,0,None identified,1


## Timestamp consistency

Timestamp-like text columns are parsed non-destructively. Invalid examples remain in the report for investigation.

In [6]:
if timestamp_issues.empty:
    print('No populated timestamp-like text columns were detected.')
else:
    display(timestamp_issues.sort_values(['invalid_values', 'file_name'], ascending=[False, True]))

,file_name,column_name,non_null_values,invalid_values,invalid_percent,invalid_examples
0,olist_order_items_dataset.csv,shipping_limit_date,112650,0,0.0,NaN
1,olist_order_reviews_dataset.csv,review_creation_date,99224,0,0.0,NaN
2,olist_order_reviews_dataset.csv,review_answer_timestamp,99224,0,0.0,NaN
3,olist_orders_dataset.csv,order_purchase_timestamp,99441,0,0.0,NaN
4,olist_orders_dataset.csv,order_approved_at,99281,0,0.0,NaN
5,olist_orders_dataset.csv,order_delivered_carrier_date,97658,0,0.0,NaN
6,olist_orders_dataset.csv,order_delivered_customer_date,96476,0,0.0,NaN
7,olist_orders_dataset.csv,order_estimated_delivery_date,99441,0,0.0,NaN


## Potential data-quality problems

These are investigation prompts generated from observed conditions. They are not automatic instructions to delete, impute, or transform data.

In [7]:
if issue_register.empty:
    print('No potential issues were flagged by the automated checks.')
else:
    display(issue_register.sort_values(['file_name', 'issue_type']).reset_index(drop=True))

,file_name,issue_type,message
0,olist_geolocation_dataset.csv,quality_warning,"Found 261,831 fully duplicated row(s)."
1,olist_geolocation_dataset.csv,quality_warning,"No single non-null, unique ID-like column was identified; verify the intended key grain."
2,olist_order_items_dataset.csv,quality_warning,"No single non-null, unique ID-like column was identified; verify the intended key grain."
3,olist_order_payments_dataset.csv,quality_warning,"No single non-null, unique ID-like column was identified; verify the intended key grain."
4,olist_order_reviews_dataset.csv,quality_warning,Found missing values in 2 column(s).
5,olist_order_reviews_dataset.csv,quality_warning,"No single non-null, unique ID-like column was identified; verify the intended key grain."
6,olist_orders_dataset.csv,business_rule,purchase_before_carrier_handoff returned status 'review'.
7,olist_orders_dataset.csv,business_rule,approval_before_carrier_handoff returned status 'review'.
8,olist_orders_dataset.csv,business_rule,carrier_handoff_before_customer_delivery returned status 'review'.
9,olist_orders_dataset.csv,quality_warning,Found missing values in 3 column(s).


## Relational contract checks

These reports validate source schemas, declared single or composite keys, and foreign-key coverage. A `review` result indicates an observed exception requiring a documented decision; it does not automatically justify deleting data.

In [8]:
print('Schema checks')
display(schema_checks)
print('Declared key checks')
display(key_checks)
print('Foreign-key coverage')
display(relationship_checks)

Schema checks


,file_name,status,missing_columns,unexpected_columns,expected_column_count,actual_column_count
0,olist_customers_dataset.csv,passed,NaN,NaN,5,5
1,olist_geolocation_dataset.csv,passed,NaN,NaN,5,5
2,olist_orders_dataset.csv,passed,NaN,NaN,8,8
3,olist_order_items_dataset.csv,passed,NaN,NaN,7,7
4,olist_order_payments_dataset.csv,passed,NaN,NaN,5,5
5,olist_order_reviews_dataset.csv,passed,NaN,NaN,7,7
6,olist_products_dataset.csv,passed,NaN,NaN,9,9
7,olist_sellers_dataset.csv,passed,NaN,NaN,4,4
8,product_category_name_translation.csv,passed,NaN,NaN,2,2


Declared key checks


,file_name,grain,primary_key,row_count,null_key_rows,duplicate_key_rows,duplicate_key_occurrences,status
0,olist_customers_dataset.csv,One marketplace customer record per customer_id,customer_id,99441,0,0,0,passed
1,olist_orders_dataset.csv,One order per order_id,order_id,99441,0,0,0,passed
2,olist_order_items_dataset.csv,One item sequence within an order,order_id + order_item_id,112650,0,0,0,passed
3,olist_order_payments_dataset.csv,One payment sequence within an order,order_id + payment_sequential,103886,0,0,0,passed
4,olist_order_reviews_dataset.csv,One review-order association,review_id + order_id,99224,0,0,0,passed
5,olist_products_dataset.csv,One product per product_id,product_id,32951,0,0,0,passed
6,olist_sellers_dataset.csv,One seller per seller_id,seller_id,3095,0,0,0,passed
7,product_category_name_translation.csv,One Portuguese product category name,product_category_name,71,0,0,0,passed


Foreign-key coverage


,relationship,child_reference,parent_reference,child_rows,null_child_keys,orphan_rows,distinct_orphan_keys,matched_non_null_percent,status
0,orders_to_customers,olist_orders_dataset.csv:customer_id,olist_customers_dataset.csv:customer_id,99441,0,0,0,100.0000,passed
1,items_to_orders,olist_order_items_dataset.csv:order_id,olist_orders_dataset.csv:order_id,112650,0,0,0,100.0000,passed
2,items_to_products,olist_order_items_dataset.csv:product_id,olist_products_dataset.csv:product_id,112650,0,0,0,100.0000,passed
3,items_to_sellers,olist_order_items_dataset.csv:seller_id,olist_sellers_dataset.csv:seller_id,112650,0,0,0,100.0000,passed
4,payments_to_orders,olist_order_payments_dataset.csv:order_id,olist_orders_dataset.csv:order_id,103886,0,0,0,100.0000,passed
5,reviews_to_orders,olist_order_reviews_dataset.csv:order_id,olist_orders_dataset.csv:order_id,99224,0,0,0,100.0000,passed
6,products_to_category_translation,olist_products_dataset.csv:product_category_name,product_category_name_translation.csv:product_category_name,32951,610,13,2,99.9598,review


## Business-rule checks

Controlled order statuses and chronological timestamp expectations are evaluated from the current source files.

In [9]:
display(
    business_rule_checks.sort_values(
        ['status', 'violation_rows', 'rule_name'],
        ascending=[True, False, True],
    ).reset_index(drop=True)
)

,rule_name,file_name,rule_type,comparable_rows,violation_rows,violation_percent,details,status
0,allowed_order_status,olist_orders_dataset.csv,allowed_values,99441,0,0.0000,NaN,passed
1,purchase_before_approval,olist_orders_dataset.csv,timestamp_order,99281,0,0.0000,order_purchase_timestamp <= order_approved_at,passed
2,purchase_before_customer_delivery,olist_orders_dataset.csv,timestamp_order,96476,0,0.0000,order_purchase_timestamp <= order_delivered_customer_date,passed
3,purchase_before_estimated_delivery,olist_orders_dataset.csv,timestamp_order,99441,0,0.0000,order_purchase_timestamp <= order_estimated_delivery_date,passed
4,review_creation_before_answer,olist_order_reviews_dataset.csv,timestamp_order,99224,0,0.0000,review_creation_date <= review_answer_timestamp,passed
5,approval_before_carrier_handoff,olist_orders_dataset.csv,timestamp_order,97644,1359,1.3918,order_approved_at <= order_delivered_carrier_date,review
6,purchase_before_carrier_handoff,olist_orders_dataset.csv,timestamp_order,97658,166,0.1700,order_purchase_timestamp <= order_delivered_carrier_date,review
7,carrier_handoff_before_customer_delivery,olist_orders_dataset.csv,timestamp_order,96475,23,0.0238,order_delivered_carrier_date <= order_delivered_customer_date,review


## Next phase

Use this audit to document table grain, expected composite keys, relationship cardinality, structural missingness, and defensible cleaning rules. Cleaning should produce new files in `data/processed`; the source CSVs remain immutable.